In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7" # Check using nvidia-smi in terminal and choose GPUs that are not being used
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from haversine import haversine
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings

import torch.nn.functional as F

def haversine_loss(pred, target):
    """
    Differentiable Haversine distance loss (in km).
    pred, target: tensors of shape [batch_size, 2] (lat, lon)
    """
    # Convert degrees to radians
    lat1, lon1 = torch.deg2rad(pred[:, 0]), torch.deg2rad(pred[:, 1])
    lat2, lon2 = torch.deg2rad(target[:, 0]), torch.deg2rad(target[:, 1])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = torch.sin(dlat / 2) ** 2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2) ** 2
    c = 2 * torch.arcsin(torch.sqrt(a))
    km = 6371 * c  # Earth's radius in kilometers
    return km.mean()

class RouterNet(pl.LightningModule):
    def __init__(self, num_classes=6, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = models.resnet18(weights="IMAGENET1K_V1")
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.criterion = nn.CrossEntropyLoss()
        self.continent_labels = ["Africa", "Asia", "Europe", "North America", "Oceania", "South America"]

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log_dict({"train_loss": loss, "train_acc": acc})
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log_dict({"val_loss": loss, "val_acc": acc}, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.hparams.lr)
    
class ResNetRegressorPLM(pl.LightningModule):
    def __init__(self, learning_rate=1e-4, backbone="resnet18", loss_type="hybrid", lambda_hav=0.001):
        """
        Args:
            learning_rate (float): optimizer learning rate
            backbone (str): 'resnet18' or 'resnet34'
            loss_type (str): 'mse', 'haversine', or 'hybrid'
            lambda_hav (float): weight for haversine loss when using hybrid
        """
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.loss_type = loss_type
        self.lambda_hav = lambda_hav

        # Choose model backbone
        if backbone == "resnet18":
            self.model = models.resnet18(weights="IMAGENET1K_V1")
        elif backbone == "resnet34":
            self.model = models.resnet34(weights="IMAGENET1K_V1")
        else:
            raise ValueError("Unsupported backbone")

        # Replace final FC layer for regression
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)

        # Define basic MSE for comparison
        self.mse = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def compute_loss(self, outputs, targets):
        """Select and compute loss based on loss_type."""
        mse_loss = self.mse(outputs, targets)
        if self.loss_type == "mse":
            return mse_loss
        elif self.loss_type == "haversine":
            return haversine_loss(outputs, targets)
        elif self.loss_type == "hybrid":
            hav_loss = haversine_loss(outputs, targets)
            return mse_loss + self.lambda_hav * hav_loss
        else:
            raise ValueError(f"Unsupported loss type: {self.loss_type}")

    def training_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.compute_loss(outputs, targets)
        rmse = torch.sqrt(self.mse(outputs, targets))
        self.log('train_loss', loss)
        self.log('train_rmse', rmse)
        return loss

    def test_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.compute_loss(outputs, targets)
        rmse = torch.sqrt(self.mse(outputs, targets))
        self.log('test_loss', loss)
        self.log('test_rmse', rmse)

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.learning_rate)

In [3]:
import torch
import torch.nn as nn

class GeoGuessrModel(nn.Module):
    def __init__(self, router_ckpt, regressor_ckpts, device="cuda"):
        """
        router_ckpt: path to trained RouterNet checkpoint
        regressor_ckpts: dict mapping continent -> ResNet checkpoint
                         e.g. {"Asia": "ckpt/Asia_resnet.ckpt", ...}
        """
        super().__init__()
        
        # Load pretrained router
        self.router = RouterNet.load_from_checkpoint(router_ckpt)
        self.router.eval().to(device)

        # Load pretrained regressors per continent
        self.regressors = {}
        for continent, ckpt_path in regressor_ckpts.items():
            regressor = ResNetRegressorPLM.load_from_checkpoint(ckpt_path)
            regressor.eval().to(device)
            self.regressors[continent] = regressor
        
        self.device = device

    def forward(self, image):
        # Predict continent
        with torch.no_grad():
            continent_logits = self.router(image)
            continent_idx = torch.argmax(continent_logits, dim=1).item()
            continent = self.router.continent_labels[continent_idx]

            # Route to correct regressor
            regressor = self.regressors.get(continent)
            coords = regressor(image)
        
        return {"continent": continent, "coords": coords}

In [4]:
geo_model = GeoGuessrModel(
    router_ckpt="outputs/router_checkpoints/RouterNet-epoch=07-val_acc=0.706.ckpt",
    regressor_ckpts={
        "Asia": "outputs/regressors/Asia_best.ckpt",
        "Europe": "outputs/regressors/Europe_best.ckpt",
        "Africa": "outputs/regressors/Africa_best.ckpt",
        # add others as you train them
    }
)

# Example: predict one image
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

img = transform(Image.open("osv-5m_subset/train/100120189161435.jpg").convert("RGB")).unsqueeze(0)
result = geo_model(img)
print(result)

FileNotFoundError: [Errno 2] No such file or directory: '/work/cssema416/202610/28/outputs/regressors/Asia_best.ckpt'